In [22]:
import torch
import torch.nn as nn

class DenseResidualBlock(nn.Module):
    def __init__(self, features: int):
        super().__init__()
        # Transformation path: F(x)
        self.fc1 = nn.Linear(features, features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(features, features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Store original input
        identity = x

        # 2. Pass through fully connected layers
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)

        # 3. Add the shortcut directly
        out = out + identity

        # 4. Final non-linearity
        return self.relu(out)




In [23]:
# --- Test with arbitrary feature vectors ---
if __name__ == "__main__":
    batch_size = 4
    num_features = 8

    # Create dummy input: 4 samples, 8 features each
    x = torch.randn(batch_size, num_features)

    model = DenseResidualBlock(features=num_features)
    output = model(x)

    print("Input shape: ", x.shape)       # torch.Size([4, 8])
    print("Output shape:", output.shape)  # torch.Size([4, 8])

Input shape:  torch.Size([4, 8])
Output shape: torch.Size([4, 8])


In [24]:
class ProjectedDenseResidualBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        # Main path
        self.fc1 = nn.Linear(in_features, out_features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(out_features, out_features)

        # Shortcut: project to match dimensions if needed
        if in_features != out_features:
            self.shortcut = nn.Linear(in_features, out_features, bias=False)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)  # Adjusts shape to [batch_size, out_features]

        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)

        # Element-wise addition now matches dimensions
        out = out + identity
        return self.relu(out)


# Example: mapping 8 features to 16 features
block = ProjectedDenseResidualBlock(in_features=8, out_features=16)
x = torch.randn(4, 8)
print("Projected Output shape:", block(x).shape)  # torch.Size([4, 16])

Projected Output shape: torch.Size([4, 16])
